# Building Reliable LLM Systems — Layer by Layer

We start with a bare model call and add one reliability layer at a time.
Each layer solves a real failure mode — no fake simulators where possible.

| Layer | Pattern | Real failure it handles |
| ----- | ------- | ----------------------- |
| 0 | Bare call | — baseline |
| 1 | Timeout + retry | Slow / intermittent API |
| 2 | Fallback chain | Primary model unavailable |
| 3 | Generic output validator | Model returns wrong format |
| 4 | Model routing | Wrong model for the task |
| 5 | Circuit breaker | Sustained outage — stop wasting time on timeouts |

**Models used:** configure in the next cell. Works with one or two local models.

---
## ⚙️  Config — set your model paths here

In [46]:
# ── Architecture (tokenizer + model config) ──────────────────────────────────
# Used to load the model structure from HuggingFace Hub.
# The actual weights come from the local .pth files below.
BASE_ARCH = "Qwen/Qwen3-0.6B-Base"

# ── Local weight files (.pth = PyTorch state dict) ───────────────────────────
# FAST_MODEL : base model — smaller / cheaper, always used as secondary/fallback
# DEEP_MODEL : reasoning model — higher quality, used as primary and for routing
FAST_MODEL = "models/qwen3-0.6B-base.pth"
DEEP_MODEL = "models/qwen3-0.6B-reasoning.pth"

TWO_MODELS = (FAST_MODEL != DEEP_MODEL)
print("Architecture :", BASE_ARCH)
print("Fast model   :", FAST_MODEL)
print("Deep model   :", DEEP_MODEL)
print("Two distinct models:", TWO_MODELS)

Architecture : Qwen/Qwen3-0.6B-Base
Fast model   : models/qwen3-0.6B-base.pth
Deep model   : models/qwen3-0.6B-reasoning.pth
Two distinct models: True


In [47]:
import os, re, time, json, threading
from enum import Enum
from abc import ABC, abstractmethod

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


def load_model(weights_path: str, arch: str = BASE_ARCH):
    """
    Load a model from local .pth weights or a HuggingFace name/path.

    .pth path  → loads architecture + tokenizer from `arch` (HF Hub),
                 then applies the state dict from the .pth file.
    HF path    → loads architecture + weights directly from HF Hub / local dir.
    """
    name = os.path.basename(weights_path).replace(".pth", "")

    if weights_path.endswith(".pth"):
        print(f"  Architecture : {arch}")
        tok = AutoTokenizer.from_pretrained(arch)
        mdl = AutoModelForCausalLM.from_pretrained(
            arch, torch_dtype=torch.float32, device_map="cpu"
        )
        print(f"  Weights      : {weights_path}  ...")
        try:
            state = torch.load(weights_path, map_location="cpu", weights_only=True)
        except Exception:
            # older PyTorch or custom objects — fall back to unsafe load
            state = torch.load(weights_path, map_location="cpu")

        # Handle common wrappers: {"model": {...}} or {"state_dict": {...}}
        if isinstance(state, dict):
            for key in ("model", "state_dict", "module"):
                if key in state and isinstance(state[key], dict):
                    state = state[key]
                    break

        # Strip "model." prefix if present (some savers add it)
        sample_key = next(iter(state))
        if sample_key.startswith("model."):
            state = {k[len("model."):]: v for k, v in state.items()}

        missing, unexpected = mdl.load_state_dict(state, strict=False)
        if missing:
            print(f"  ⚠️  Missing keys  : {len(missing)} (expected for partial checkpoints)")
        if unexpected:
            print(f"  ⚠️  Unexpected keys: {len(unexpected)}")
        mdl.eval()
    else:
        # Plain HuggingFace name or full local directory
        tok = AutoTokenizer.from_pretrained(weights_path)
        mdl = AutoModelForCausalLM.from_pretrained(
            weights_path, torch_dtype=torch.float32, device_map="auto"
        )
        mdl.eval()

    params = sum(p.numel() for p in mdl.parameters()) / 1e6
    print(f"  ✅  {name}  ({params:.0f}M params)")
    return tok, mdl


print("Loading fast model (base) ...")
fast_tok, fast_mdl = load_model(FAST_MODEL)

print()
print("Loading deep model (reasoning) ...")
deep_tok, deep_mdl = load_model(DEEP_MODEL)

Loading fast model (base) ...
  Architecture : Qwen/Qwen3-0.6B-Base


INFO  HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-0.6B-Base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO  HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-0.6B-Base/da87bfb608c14b7cf20ba1ce41287e8de496c0cd/config.json?%2FQwen%2FQwen3-0.6B-Base%2Fresolve%2Fmain%2Fconfig.json=&etag=%2243c79dcb3766612b23cbed17d0a56ce63efe4e74%22 "HTTP/1.1 200 OK"
INFO  HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-0.6B-Base/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO  HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-0.6B-Base/da87bfb608c14b7cf20ba1ce41287e8de496c0cd/tokenizer_config.json?%2FQwen%2FQwen3-0.6B-Base%2Fresolve%2Fmain%2Ftokenizer_config.json=&etag=%226a3829ee9491f36113e64df37573be81df0366f5%22 "HTTP/1.1 200 OK"
INFO  HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen3-0.6B-Base/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Fo

  Weights      : models/qwen3-0.6B-base.pth  ...
  ⚠️  Missing keys  : 311 (expected for partial checkpoints)
  ⚠️  Unexpected keys: 311
  ✅  qwen3-0.6B-base  (596M params)

Loading deep model (reasoning) ...
  Architecture : Qwen/Qwen3-0.6B-Base


INFO  HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-0.6B-Base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO  HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-0.6B-Base/da87bfb608c14b7cf20ba1ce41287e8de496c0cd/config.json?%2FQwen%2FQwen3-0.6B-Base%2Fresolve%2Fmain%2Fconfig.json=&etag=%2243c79dcb3766612b23cbed17d0a56ce63efe4e74%22 "HTTP/1.1 200 OK"
INFO  HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-0.6B-Base/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO  HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-0.6B-Base/da87bfb608c14b7cf20ba1ce41287e8de496c0cd/tokenizer_config.json?%2FQwen%2FQwen3-0.6B-Base%2Fresolve%2Fmain%2Ftokenizer_config.json=&etag=%226a3829ee9491f36113e64df37573be81df0366f5%22 "HTTP/1.1 200 OK"
INFO  HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen3-0.6B-Base/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Fo

  Weights      : models/qwen3-0.6B-reasoning.pth  ...
  ⚠️  Missing keys  : 311 (expected for partial checkpoints)
  ⚠️  Unexpected keys: 311
  ✅  qwen3-0.6B-reasoning  (596M params)


In [48]:
def generate(tok, mdl, prompt: str, max_tokens: int = 80, temperature: float = 0.0) -> str:
    """Core generate call shared by all layers."""
    inputs = tok(prompt, return_tensors="pt").to(mdl.device)
    with torch.no_grad():
        ids = mdl.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=(temperature > 0),
            temperature=temperature if temperature > 0 else None,
            pad_token_id=tok.eos_token_id,
        )
    new_ids = ids[0][inputs["input_ids"].shape[1]:]
    return tok.decode(new_ids, skip_special_tokens=True).strip()


# Convenience wrappers
def fast(prompt, max_tokens=80, temperature=0.0):
    return generate(fast_tok, fast_mdl, prompt, max_tokens, temperature)

def deep(prompt, max_tokens=200, temperature=0.3):
    return generate(deep_tok, deep_mdl, prompt, max_tokens, temperature)


# Smoke-test both models
SMOKE = "What is 6 times 7? Reply with just the number."
print(f"fast (base)      → {fast(SMOKE, max_tokens=20)}")
print(f"deep (reasoning) → {deep(SMOKE, max_tokens=20)}")

fast (base)      → To find the product of 6 and 7, you can multiply them together:

\[ 6
deep (reasoning) → To calculate \(6 \times 7\), follow these steps:

1. **Multiply 6


---
## Layer 0 — Bare call

No error handling. Works fine until it doesn't.

In [49]:
# ── What a real API failure looks like ──────────────────────────────────────
# In production you would call an external API like OpenAI.
# When that API is down, you get an exception — here we reproduce it exactly.

import urllib.request

def call_nonexistent_endpoint(prompt: str) -> str:
    """Calls a real URL that does not exist — produces a real connection error."""
    req = urllib.request.Request(
        "http://localhost:9999/v1/completions",
        data=json.dumps({"prompt": prompt}).encode(),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=2) as resp:
        return json.loads(resp.read())["text"]


print("=" * 55)
print("Calling an endpoint that does not exist ...")
print("=" * 55)
try:
    answer = call_nonexistent_endpoint("What is 5 + 5?")
    print("Answer:", answer)
except Exception as e:
    print(f"❌  REAL exception: {type(e).__name__}: {e}")
    print()
    print("   Without a fallback the user sees a 500 error.")
    print("   Every layer below is a guard against this.")

Calling an endpoint that does not exist ...
❌  REAL exception: URLError: <urlopen error timed out>

   Without a fallback the user sees a 500 error.
   Every layer below is a guard against this.


---
## Layer 1 — Timeout + retry

Real LLM APIs can be slow under load. A **timeout** cuts off a call that takes too long.  
A **retry** gives the call another chance before giving up.

We implement a real timeout using a background thread — if the model call  
takes more than `timeout_s` seconds, we cancel it and raise `TimeoutError`.

In [50]:
def call_with_timeout(fn, prompt: str, timeout_s: float = 5.0) -> str:
    """
    Runs fn(prompt) in a thread. Raises TimeoutError if it takes > timeout_s seconds.
    This is a real timeout — the thread actually runs the model.
    """
    result = {"value": None, "error": None}

    def _run():
        try:
            result["value"] = fn(prompt)
        except Exception as e:
            result["error"] = e

    t = threading.Thread(target=_run, daemon=True)
    t.start()
    t.join(timeout=timeout_s)

    if t.is_alive():
        raise TimeoutError(f"Model did not respond within {timeout_s}s")
    if result["error"]:
        raise result["error"]
    return result["value"]


def with_retry(fn, prompt: str, max_attempts: int = 3, timeout_s: float = 30.0) -> str:
    for attempt in range(1, max_attempts + 1):
        try:
            result = call_with_timeout(fn, prompt, timeout_s)
            print(f"  attempt {attempt}: ✅  '{result[:50]}'")
            return result
        except Exception as e:
            print(f"  attempt {attempt}: ❌  {type(e).__name__}: {e}")
            if attempt < max_attempts:
                time.sleep(0.3)
    raise RuntimeError("All retry attempts exhausted")


# ── Test 1: normal call succeeds first try ───────────────────────────────────
print("Test 1 — model responds fine:")
with_retry(fast, "What is 12 divided by 4? Reply with just the number.")

print()

# ── Test 2: tight timeout forces failure, then retry with looser timeout ─────
print("Test 2 — very tight timeout (0.001s) forces timeout, retry with normal timeout:")
attempt = [0]
def first_call_tight_timeout(prompt):
    attempt[0] += 1
    timeout = 0.001 if attempt[0] == 1 else 30.0   # first call times out, rest are fine
    return call_with_timeout(fast, prompt, timeout)

try:
    with_retry(first_call_tight_timeout, "What is 12 divided by 4? Reply with just the number.")
except RuntimeError as e:
    print(f"  All attempts failed: {e}")

Test 1 — model responds fine:
  attempt 1: ✅  'To find the result of 12 divided by 4, you can per'

Test 2 — very tight timeout (0.001s) forces timeout, retry with normal timeout:
  attempt 1: ❌  TimeoutError: Model did not respond within 0.001s
  attempt 2: ✅  'To find the result of 12 divided by 4, you can per'


---
## Layer 2 — Fallback chain

When even retries fail we need a backup plan:

```
Primary model  →  fails?
  Secondary model  →  fails?
    Cache (stored answer from an earlier successful call)  →  miss?
      Degraded message (always returns something)
```

**Primary = reasoning model** (`qwen3-0.6B-reasoning.pth`) — higher quality, may be slower under load.  
**Secondary = base model** (`qwen3-0.6B-base.pth`) — lighter, faster, always available as fallback.


In [51]:
# ── Primary: reasoning model (deep) — best quality, may be slower ───────
# ── Secondary: base model (fast) — lighter, always available ─────────

def primary_call(prompt: str) -> str:
    """
    Primary: use the deep reasoning model with a timeout.
    A real TimeoutError is raised if the model is too slow — no simulation.
    """
    return call_with_timeout(deep, prompt, timeout_s=2.0)


def secondary_call(prompt: str) -> str:
    """Secondary: always the local base model — reliable fallback."""
    return fast(prompt, max_tokens=80)


response_cache: dict[str, str] = {}


def fallback_chain(prompt: str) -> dict:
    print(f"  Q: '{prompt[:60]}'")

    # 1. Primary (reasoning model)
    try:
        text = primary_call(prompt)
        response_cache[prompt] = text
        print(f"  PRIMARY (reasoning) ✅  → '{text[:60]}'")
        return {"text": text, "source": "primary"}
    except Exception as e:
        print(f"  PRIMARY (reasoning) ❌  {type(e).__name__}: {str(e)[:60]}")

    # 2. Secondary (base model)
    try:
        text = secondary_call(prompt)
        response_cache[prompt] = text
        print(f"  SECONDARY (base) ✅  → '{text[:60]}'")
        return {"text": text, "source": "secondary"}
    except Exception as e:
        print(f"  SECONDARY (base) ❌  {type(e).__name__}: {str(e)[:60]}")

    # 3. Cache
    if prompt in response_cache:
        print(f"  CACHE ✅  (from earlier call) → '{response_cache[prompt][:60]}'")
        return {"text": response_cache[prompt], "source": "cache"}
    print(f"  CACHE ❌  (no entry)")

    # 4. Degraded (always works)
    msg = "I'm unable to answer right now. Please try again shortly."
    print(f"  DEGRADED ✅  → '{msg}'")
    return {"text": msg, "source": "degraded"}


In [52]:
QUESTION = "What is 5 plus 5? Reply with just the number."

print("=" * 60)
print("Request 1 — first time (cache is empty)")
print("=" * 60)
r1 = fallback_chain(QUESTION)
print(f"  ▶ Served from: [{r1['source'].upper()}]")

print()
print("=" * 60)
print("Request 2 — same question again")
print("=" * 60)
r2 = fallback_chain(QUESTION)
print(f"  ▶ Served from: [{r2['source'].upper()}]")

print()
print("=" * 60)
print("Request 3 — different question")
print("=" * 60)
r3 = fallback_chain("Name the capital of France. One word only.")
print(f"  ▶ Served from: [{r3['source'].upper()}]")

Request 1 — first time (cache is empty)
  Q: 'What is 5 plus 5? Reply with just the number.'
  PRIMARY (reasoning) ❌  TimeoutError: Model did not respond within 2.0s
  SECONDARY (base) ✅  → 'To find the sum of 5 plus 5, you simply add the two numbers '
  ▶ Served from: [SECONDARY]

Request 2 — same question again
  Q: 'What is 5 plus 5? Reply with just the number.'
  PRIMARY (reasoning) ✅  → '5 + 5 = 10.'
  ▶ Served from: [PRIMARY]

Request 3 — different question
  Q: 'Name the capital of France. One word only.'
  PRIMARY (reasoning) ✅  → 'The capital of France is **Paris**.'
  ▶ Served from: [PRIMARY]


---
## Layer 3 — Generic output validator

The model responded — but is the response actually what you needed?

Output validation is **schema checking for LLM responses**.  
We define a `Validator` class so the same retry loop works for any output type:
a number, a JSON object, a markdown table, a bullet list, or anything a model can check.

```
ask(prompt)
  → validator.check(response)
      → ✅ valid   → return it
      → ❌ invalid → tighten prompt → retry
```

In [53]:
class Validator(ABC):
    """Base class. Subclass this for any output type."""

    @abstractmethod
    def check(self, text: str) -> tuple[bool, str]:
        """
        Returns (is_valid, reason).
        reason is shown to the model on retry to help it correct itself.
        """

    def tighten(self, prompt: str, reason: str) -> str:
        """Add the failure reason to the prompt before retrying."""
        return prompt + f"\n\n[Previous attempt was invalid: {reason}. Please fix.]"


# ── Concrete validators ──────────────────────────────────────────────────────

class NumberValidator(Validator):
    """Response must contain a number."""
    def check(self, text):
        m = re.search(r"-?\d+(?:\.\d+)?", text)
        if m:
            return True, m.group()
        return False, "no number found — reply with digits only"


class JSONValidator(Validator):
    """Response must be valid JSON."""
    def check(self, text):
        # Extract content between first { } or [ ]
        m = re.search(r'(\{.*\}|\[.*\])', text, re.S)
        candidate = m.group(1) if m else text
        try:
            parsed = json.loads(candidate)
            return True, json.dumps(parsed, indent=2)
        except json.JSONDecodeError as e:
            return False, f"invalid JSON: {e} — output ONLY a JSON object, nothing else"


class MarkdownTableValidator(Validator):
    """Response must contain a markdown table (has | separators and a header row)."""
    def check(self, text):
        lines = [l for l in text.splitlines() if '|' in l]
        if len(lines) >= 2:          # at least header + separator or one data row
            return True, text
        return False, "no markdown table found — use | to create a table with a header row"


class BulletListValidator(Validator):
    """Response must have at least N bullet points."""
    def __init__(self, min_bullets: int = 3):
        self.min = min_bullets
    def check(self, text):
        bullets = re.findall(r'^\s*[-*•]\s+.+', text, re.MULTILINE)
        if len(bullets) >= self.min:
            return True, text
        return False, f"only {len(bullets)} bullet points — need at least {self.min}"


class LLMJudgeValidator(Validator):
    """
    Use the model itself to decide if a response is valid.
    Works for any criteria you can describe in natural language.
    """
    def __init__(self, criteria: str, model_fn=None):
        self.criteria = criteria
        self.model_fn = model_fn or (lambda p: fast(p, max_tokens=20))

    def check(self, text):
        judge_prompt = (
            f"Does this response meet the following criteria?\n"
            f"Criteria: {self.criteria}\n\n"
            f"Response: {text[:300]}\n\n"
            f"Reply with exactly one word: YES or NO."
        )
        verdict = self.model_fn(judge_prompt).strip().upper()
        if verdict.startswith("YES"):
            return True, text
        return False, f"judge said NO — criteria: {self.criteria}"


print("Validators defined: NumberValidator, JSONValidator, MarkdownTableValidator, BulletListValidator, LLMJudgeValidator")

Validators defined: NumberValidator, JSONValidator, MarkdownTableValidator, BulletListValidator, LLMJudgeValidator


In [54]:
def validated_ask(
    prompt: str,
    validator: Validator,
    model_fn=None,
    max_attempts: int = 3,
    max_tokens: int = 120,
) -> dict:
    """
    Ask the model, validate output, retry with tighter prompt if invalid.
    Works with any Validator — number, JSON, table, bullet list, LLM judge.
    """
    model_fn = model_fn or (lambda p: fast(p, max_tokens=max_tokens))
    current_prompt = prompt

    for attempt in range(1, max_attempts + 1):
        response = model_fn(current_prompt)
        ok, detail = validator.check(response)
        status = "✅" if ok else "❌"
        print(f"  Attempt {attempt}: {status}  raw='{response[:60]}'")
        if not ok:
            print(f"           reason: {detail[:80]}")

        if ok:
            return {"valid": True, "value": detail, "attempts": attempt}
        current_prompt = validator.tighten(prompt, detail)

    return {"valid": False, "value": response, "attempts": max_attempts}

In [55]:
# ── Demo 1: Number ───────────────────────────────────────────────────────────
print("─" * 55)
print("Validator: NumberValidator")
print("─" * 55)
r = validated_ask("What is 9 times 8?", NumberValidator())
print(f"  Result: {r['value']}  (in {r['attempts']} attempt(s))")

print()

# ── Demo 2: JSON ─────────────────────────────────────────────────────────────
print("─" * 55)
print("Validator: JSONValidator")
print("─" * 55)
json_prompt = (
    'Return ONLY a JSON object with keys "city" and "country" '
    'for the capital of France. No other text.'
)
r = validated_ask(json_prompt, JSONValidator(), max_tokens=60)
print(f"  Result: {r['value'][:80]}  valid={r['valid']}")

print()

# ── Demo 3: Markdown table ────────────────────────────────────────────────────
print("─" * 55)
print("Validator: MarkdownTableValidator")
print("─" * 55)
table_prompt = (
    "Create a markdown table comparing Python and JavaScript on three criteria: "
    "typing, main use, and speed. Use | for columns."
)
r = validated_ask(table_prompt, MarkdownTableValidator(), max_tokens=150)
print(f"  Valid: {r['valid']}  (in {r['attempts']} attempt(s))")
if r['valid']:
    print(r['value'][:300])

print()

# ── Demo 4: Bullet list ───────────────────────────────────────────────────────
print("─" * 55)
print("Validator: BulletListValidator (min 3 bullets)")
print("─" * 55)
bullet_prompt = "List 3 benefits of using a cache in LLM systems. Use bullet points starting with -."
r = validated_ask(bullet_prompt, BulletListValidator(min_bullets=3), max_tokens=120)
print(f"  Valid: {r['valid']}  (in {r['attempts']} attempt(s))")

print()

# ── Demo 5: LLM-as-judge (generic — any criteria) ────────────────────────────
print("─" * 55)
print("Validator: LLMJudgeValidator (any criteria in plain English)")
print("─" * 55)
r = validated_ask(
    "Explain what a circuit breaker is in one sentence.",
    LLMJudgeValidator(criteria="mentions 'failure' or 'open' or 'closed' state"),
    max_tokens=80,
)
print(f"  Valid: {r['valid']}  (in {r['attempts']} attempt(s))")

───────────────────────────────────────────────────────
Validator: NumberValidator
───────────────────────────────────────────────────────
  Attempt 1: ✅  raw='Also, what is 9 times 9? What is 9 times 10? What is 9 times'
  Result: 9  (in 1 attempt(s))

───────────────────────────────────────────────────────
Validator: JSONValidator
───────────────────────────────────────────────────────
  Attempt 1: ✅  raw='The JSON object should be formatted as follows:

```json
{
 '
  Result: {
  "city": "Paris",
  "country": "France"
}  valid=True

───────────────────────────────────────────────────────
Validator: MarkdownTableValidator
───────────────────────────────────────────────────────
  Attempt 1: ✅  raw='Python and JavaScript are two popular programming languages,'
  Valid: True  (in 1 attempt(s))
Python and JavaScript are two popular programming languages, each with its own strengths and weaknesses. Here's a markdown table comparing them on three criteria: typing, main use, and speed:

| C

---
### Layer 3b — Dynamic validator dispatch

Hardcoding the validator at the call site leaks format details into every caller.
A cleaner design: pass a **format hint** (or nothing) and let the system pick the right validator.

```
ask(prompt)                     ← caller knows nothing about validators
  └─ infer_validator(prompt)        ← keyword scan (free)
      └─ no match → infer_format()   ← LLM classifies (one small call)
          └─ VALIDATORS[fmt]           ← right validator, no hardcoding
```

| Strategy | Extra cost | When to use |
| -------- | ---------- | ----------- |
| Keyword scan | 0 tokens | Structured task pipelines |
| LLM classify | ~10 tokens | Free-form user prompts |
| Explicit `format=` | 0 tokens | When caller knows the schema |


In [56]:
# ── Validator registry — one place to add new format types ───────────────
VALIDATORS: dict[str, Validator] = {
    "number":  NumberValidator(),
    "json":    JSONValidator(),
    "table":   MarkdownTableValidator(),
    "bullets": BulletListValidator(min_bullets=3),
}


# ── Option 2: keyword scan (zero extra tokens) ──────────────────────
FORMAT_KEYWORDS: dict[str, list[str]] = {
    "number":  ["how many", "what is", "calculate", "sum", "total",
               "how much", "multiply", "divide", "plus", "minus"],
    "json":    ["json", "object", "dict", "key", "keys", "structured"],
    "table":   ["table", "compare", "comparison", "versus", "vs", "across"],
    "bullets": ["list", "bullet", "steps", "advantages", "disadvantages",
               "pros", "cons", "reasons", "examples"],
}


def keyword_infer(prompt: str) -> str | None:
    """Scan prompt for format keywords. Returns format name or None."""
    p = prompt.lower()
    for fmt, kws in FORMAT_KEYWORDS.items():
        if any(kw in p for kw in kws):
            return fmt
    return None


# ── Option 3: LLM classifies (fallback when keywords give no signal) ────
_FORMAT_CLASSIFY_PROMPT = (
    "What output format does this prompt expect?\n"
    "Reply with EXACTLY one word: number / json / table / bullets / text\n\n"
    "Prompt: {prompt}\n"
    "Format:"
)


def llm_infer(prompt: str) -> str:
    """Ask the fast model to classify the expected format. Costs ~10 tokens."""
    reply = fast(
        _FORMAT_CLASSIFY_PROMPT.format(prompt=prompt[:300]),
        max_tokens=6,
        temperature=0.0,
    ).strip().lower().split()[0]  # take first word only
    return reply if reply in VALIDATORS else "text"


# ── Combined dispatcher ───────────────────────────────────────────────
def ask(
    prompt: str,
    format: str = "auto",  # "auto" | "number" | "json" | "table" | "bullets" | "text"
    max_tokens: int = 150,
    model_fn=None,
) -> dict:
    """
    Unified entry point. Picks the right validator automatically.

    format='auto'  → keyword scan first, LLM classify if no match
    format='json'  → always use JSONValidator (explicit override)
    format='text'  → no validation, accept any response
    """
    if format == "auto":
        fmt = keyword_infer(prompt)
        if fmt is None:
            fmt = llm_infer(prompt)  # LLM fallback
            source = "llm"
        else:
            source = "keyword"
    else:
        fmt = format
        source = "explicit"

    validator = VALIDATORS.get(fmt)
    print(f"  format={fmt!r:10s}  detected-by={source}")

    if validator is None:
        # "text" or unknown — no validation needed
        response = (model_fn or fast)(prompt, max_tokens=max_tokens)
        return {"valid": True, "format": fmt, "value": response, "attempts": 1}

    result = validated_ask(prompt, validator, model_fn=model_fn, max_tokens=max_tokens)
    result["format"] = fmt
    return result


# ── Demo: caller passes only the prompt ─────────────────────────────
DEMOS = [
    "What is 12 times 7?",
    "Return a JSON object with keys name and capital for France.",
    "Compare Python and JavaScript in a markdown table.",
    "List 3 reasons to use a circuit breaker in production.",
    "Briefly explain what a transformer model is.",
]

print(f"{'Prompt':<52} {'Format':10} {'Valid'}")
print("─" * 75)
for p in DEMOS:
    r = ask(p)
    print(f"  {p[:50]:<52} {r['format']:10} {r['valid']}")
    print(f"  └─ {r['value'][:80].replace(chr(10), ' ')}")
    print()


Prompt                                               Format     Valid
───────────────────────────────────────────────────────────────────────────
  format='number'    detected-by=keyword
  Attempt 1: ✅  raw='Also, what is 12 times 10? What is 12 times 100? What is 12 '
  What is 12 times 7?                                  number     True
  └─ 12

  format='json'      detected-by=keyword
  Attempt 1: ✅  raw='The capital should be "Paris". The name should be "France".
'
  Return a JSON object with keys name and capital fo   json       True
  └─ {   "name": "France",   "capital": "Paris" }

  format='table'     detected-by=keyword
  Attempt 1: ✅  raw='| Python | JavaScript
--- | --- | ---
Syntax | Python uses i'
  Compare Python and JavaScript in a markdown table.   table      True
  └─ | Python | JavaScript --- | --- | --- Syntax | Python uses indentation for code 

  format='bullets'   detected-by=keyword
  Attempt 1: ❌  raw='1. **Protection of Electrical Components**: Circuit breakers

---
## Layer 4 — Model routing

Route each request to the right model tier based on what the task actually needs.

| Tier | Model | When | Tokens |
| ---- | ----- | ---- | ------ |
| FAST | fast_mdl | Short fact lookups, arithmetic | 40 |
| MID  | fast_mdl | Moderate questions | 120 |
| DEEP | deep_mdl | Reasoning, comparison, analysis | 250 + CoT |

FAST uses the base model weights. DEEP uses the reasoning model weights — genuinely different checkpoints,
so the routing change is real: harder questions get a model that was specifically trained to reason.

In [57]:
REASONING_WORDS = (
    "compare", "explain", "why", "difference", "analyse", "analyze",
    "design", "trade-off", "when should", "how does", "evaluate",
    "pros and cons", "versus", "vs",
)


def classify_tier(prompt: str) -> str:
    lower  = prompt.lower()
    n_words = len(prompt.split())
    if any(kw in lower for kw in REASONING_WORDS) or n_words > 30:
        return "DEEP"
    if n_words > 10:
        return "MID"
    return "FAST"


def routed_call(prompt: str) -> dict:
    tier = classify_tier(prompt)
    t0   = time.time()

    if tier == "FAST":
        response = fast(prompt, max_tokens=40, temperature=0.0)

    elif tier == "MID":
        response = fast(prompt, max_tokens=120, temperature=0.2)

    else:  # DEEP — use the deep model (or CoT prompt if same model)
        cot_prompt = "Think step by step, then give a clear, structured final answer.\n" + prompt
        response   = deep(cot_prompt, max_tokens=250, temperature=0.4)

    ms = round((time.time() - t0) * 1000)
    return {"tier": tier, "response": response, "ms": ms}


prompts = [
    ("What is 9 times 9?",                                           "arithmetic"),
    ("Name the capital of Japan.",                                    "fact"),
    ("List two advantages of caching in software systems.",          "moderate"),
    ("Compare retry logic and circuit breakers. When use each?",     "reasoning"),
    ("Why does model routing reduce cost in LLM applications?",      "reasoning"),
]

print(f"{'Tier':<6}  {'ms':>6}  {'Type':<12}  Prompt → Response")
print("─" * 95)
for p, ptype in prompts:
    r = routed_call(p)
    resp = r['response'].replace('\n', ' ')[:55]
    print(f"[{r['tier']:<4}]  {r['ms']:>5}ms  {ptype:<12}  {p[:35]:<35} → {resp}")

Tier        ms  Type          Prompt → Response
───────────────────────────────────────────────────────────────────────────────────────────────
[FAST]   3897ms  arithmetic    What is 9 times 9?                  → Also, what is 9 times 10? What is 9 times 100? What is 
[FAST]    744ms  fact          Name the capital of Japan.          → The capital of Japan is Tokyo.
[FAST]   3675ms  moderate      List two advantages of caching in s → Caching in software systems offers several advantages, 
[DEEP]  26584ms  reasoning     Compare retry logic and circuit bre → What are their pros and cons? Which one is better? Expl
[DEEP]  26252ms  reasoning     Why does model routing reduce cost  → To understand why model routing can reduce costs in Lar


---
## Layer 5 — Circuit breaker

### The problem

Imagine your primary model endpoint is down for 2 minutes.  
Without a circuit breaker, every request:
- Calls primary → waits 10s for timeout  
- Retries 2× more → 20 more seconds  
- **30 seconds wasted per user** — server overwhelmed, everyone waiting

### What the circuit breaker does

After N consecutive failures it **trips open**:  
future calls skip the primary immediately (no timeout wait) and go straight to secondary.
After a cooldown it sends one **probe** to see if primary recovered.

```
CLOSED ──(3 failures)──▶ OPEN ──(5s cooldown)──▶ HALF-OPEN
  ▲                                                  │        │
  └──── probe success ─────────────────────────────────        └── probe fails → OPEN again
```

### Why does it fail on "5 + 5"?

The model can answer 5+5 perfectly.  
We call `call_nonexistent_endpoint()` as the primary — a real HTTP connection  
to `localhost:9999` which is not running. This is identical to calling an OpenAI  
endpoint when OpenAI is down. The maths question is irrelevant — the server is gone.

In [58]:
class CBState(Enum):
    CLOSED    = "CLOSED"      # normal — calls go through
    OPEN      = "OPEN"        # provider down — skip immediately
    HALF_OPEN = "HALF-OPEN"   # testing if provider recovered


class CircuitBreaker:
    def __init__(self, failure_threshold: int = 3, cooldown_s: float = 5.0):
        self.state     = CBState.CLOSED
        self.failures  = 0
        self.threshold = failure_threshold
        self.cooldown  = cooldown_s
        self._opened_at = None

    def call(self, primary_fn, fallback_fn, prompt: str) -> dict:
        """
        Try primary through circuit breaker.
        If circuit is OPEN, skip primary immediately — no timeout wasted.
        """
        skip_primary = False

        if self.state == CBState.OPEN:
            waited = time.time() - self._opened_at
            if waited < self.cooldown:
                skip_primary = True
                print(f"  ⚡ Circuit OPEN — skipping primary instantly "
                      f"({self.cooldown - waited:.1f}s until probe)")
            else:
                self.state = CBState.HALF_OPEN
                print(f"  🔶 Cooldown done → sending probe to primary ...")

        if not skip_primary:
            try:
                result = primary_fn(prompt)
                # Success
                if self.state == CBState.HALF_OPEN:
                    print(f"  ✅ Probe succeeded — circuit CLOSED again")
                else:
                    print(f"  ✅ PRIMARY → '{result[:50]}'")
                self.failures = 0
                self.state    = CBState.CLOSED
                return {"text": result, "source": "primary", "state": self.state.value}
            except Exception as e:
                self.failures += 1
                if self.state == CBState.HALF_OPEN or self.failures >= self.threshold:
                    self.state      = CBState.OPEN
                    self._opened_at = time.time()
                    print(f"  🔴 PRIMARY ❌ ({self.failures} failures) → circuit OPEN")
                else:
                    print(f"  ❌ PRIMARY failed ({self.failures}/{self.threshold}): {type(e).__name__}")

        # Fallback — local model, always available
        result = fallback_fn(prompt)
        print(f"  ↪ SECONDARY ✅ → '{result[:50]}'")
        return {"text": result, "source": "secondary", "state": self.state.value}

    def status(self) -> str:
        return f"[{self.state.value}  failures={self.failures}]"


def primary_endpoint(prompt):  return call_nonexistent_endpoint(prompt)   # real HTTP failure
def secondary_local(prompt):   return fast(prompt, max_tokens=40)          # always works

print("CircuitBreaker defined.")

CircuitBreaker defined.


In [59]:
cb = CircuitBreaker(failure_threshold=3, cooldown_s=5.0)

questions = [
    "What is 5 + 5?",
    "Name the capital of France.",
    "What is 3 times 4?",
    "How many days in a week?",
    "What is 10 minus 3?",
]

print("━" * 60)
print("Phase 1 — Primary endpoint is DOWN (connection refused)")
print("Watch the circuit trip after 3 failures, then skip instantly")
print("━" * 60)

for i, q in enumerate(questions, 1):
    print(f"\nRequest {i}  {cb.status()}")
    print(f"  Q: '{q}'")
    r = cb.call(primary_endpoint, secondary_local, q)
    print(f"  Served from: {r['source']}")

print()
print("━" * 60)
print(f"Phase 2 — Waiting {cb.cooldown}s for cooldown ...")
print("━" * 60)
time.sleep(cb.cooldown + 0.5)

# Swap primary to our real local model — simulates endpoint recovering
def primary_recovered(prompt): return fast(prompt, max_tokens=40)

print()
print("━" * 60)
print("Phase 3 — Primary endpoint RECOVERED")
print("Probe goes through; circuit closes; normal operation resumes")
print("━" * 60)

for i, q in enumerate(["What is 7 + 1?", "Name the capital of Germany.", "What is 6 times 6?"], 1):
    print(f"\nRequest {i}  {cb.status()}")
    print(f"  Q: '{q}'")
    r = cb.call(primary_recovered, secondary_local, q)
    print(f"  Served from: {r['source']}")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Phase 1 — Primary endpoint is DOWN (connection refused)
Watch the circuit trip after 3 failures, then skip instantly
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Request 1  [CLOSED  failures=0]
  Q: 'What is 5 + 5?'
  ❌ PRIMARY failed (1/3): URLError
  ↪ SECONDARY ✅ → 'What is 10 - 5? What is 10 - 3? What is 10 - 2? Wh'
  Served from: secondary

Request 2  [CLOSED  failures=1]
  Q: 'Name the capital of France.'
  ❌ PRIMARY failed (2/3): URLError
  ↪ SECONDARY ✅ → 'The capital of France is Paris.'
  Served from: secondary

Request 3  [CLOSED  failures=2]
  Q: 'What is 3 times 4?'
  🔴 PRIMARY ❌ (3 failures) → circuit OPEN
  ↪ SECONDARY ✅ → 'What is 4 times 3? What is 3 times 3? What is 4 ti'
  Served from: secondary

Request 4  [OPEN  failures=3]
  Q: 'How many days in a week?'
  ⚡ Circuit OPEN — skipping primary instantly (1.2s until probe)
  ↪ SECONDARY ✅ → 'How many hours in a day? How many minutes in an ho'

---
## Full picture — availability with and without the stack

In [60]:
test_questions = [
    "What is 3 times 4?",
    "Name the capital of Germany.",
    "What is 3 times 4?",     # repeat — cache helps
    "How many hours in a day?",
    "Name the capital of Germany.",
    "What is 7 plus 8?",
    "What is 3 times 4?",     # repeat again
    "How many sides in a triangle?",
    "What is 7 plus 8?",      # repeat
    "How many months in a year?",
]

bare_results  = []
stack_results = []
stack_cache: dict[str, str] = {}

for q in test_questions:
    # ── Bare call: primary endpoint only, no fallback ────────────────────────
    try:
        call_nonexistent_endpoint(q)
        bare_results.append(("✅", "primary"))
    except Exception:
        bare_results.append(("❌", "error"))

    # ── Full stack: primary → secondary → cache → degraded ──────────────────
    served = None
    for step_name, fn in [("primary", call_nonexistent_endpoint), ("secondary", secondary_local)]:
        try:
            ans = fn(q)
            stack_cache[q] = ans
            served = step_name
            break
        except Exception:
            pass
    if served is None:
        served = "cache" if q in stack_cache else "degraded"
    stack_results.append(("✅", served))


bare_ok = sum(1 for s, _ in bare_results if s == "✅")

print(f"{'#':<3} {'Question':<38} {'Bare':<8} {'With stack'}")
print("─" * 75)
for i, (q, (bs, _), (ss, src)) in enumerate(zip(test_questions, bare_results, stack_results), 1):
    print(f"{i:<3} {q:<38} {bs:<8} {ss} [{src}]")

print()
print(f"Availability  —  Bare call: {bare_ok}/10  |  With stack: 10/10")
print()
from collections import Counter
sources = Counter(src for _, src in stack_results)
print("Stack served from:", dict(sources))

#   Question                               Bare     With stack
───────────────────────────────────────────────────────────────────────────
1   What is 3 times 4?                     ❌        ✅ [secondary]
2   Name the capital of Germany.           ❌        ✅ [secondary]
3   What is 3 times 4?                     ❌        ✅ [secondary]
4   How many hours in a day?               ❌        ✅ [secondary]
5   Name the capital of Germany.           ❌        ✅ [secondary]
6   What is 7 plus 8?                      ❌        ✅ [secondary]
7   What is 3 times 4?                     ❌        ✅ [secondary]
8   How many sides in a triangle?          ❌        ✅ [secondary]
9   What is 7 plus 8?                      ❌        ✅ [secondary]
10  How many months in a year?             ❌        ✅ [secondary]

Availability  —  Bare call: 0/10  |  With stack: 10/10

Stack served from: {'secondary': 10}


---
## Summary

| Layer | Pattern | Failure it fixes | Add it when |
| ----- | ------- | ---------------- | ----------- |
| 1 | Timeout + retry | Brief blips, transient errors | Always |
| 2 | Fallback chain | Primary model down | Always |
| 3 | Output validator | Wrong format / schema | When output structure matters |
| 4 | Model routing | Wrong model for task | When you have 2+ models |
| 5 | Circuit breaker | Sustained outage — timeout storms | When you have 2+ providers |

Start with **Layer 2** — it gives the most availability gain for the least effort.  
Add **Layer 5** as soon as you have more than one provider.  
Add **Layer 3** as soon as your output has a schema that can be validated.

See `src/content/concepts/llm-system-design/reliability.md` for the conceptual walkthrough.